In [1]:
import math

from steel_lib.si_units import si
from steelpy import aisc
from steel_lib.connection_factory import ConnectionFactory
from steel_lib.data_models import (
    Plate,
    ConnectionComponent,
    DesignLoads,GlobalLoads,BoltConfiguration,
  
)
from steel_lib.materials import MATERIALS, BOLT_GRADES, WELD_ELECTRODES
from steel_lib.member_factory import MemberFactory
from steel_lib.calculations import (
    BoltShearCalculator,
    BlockShearCalculator,
    ConnectionCapacityCalculator,
    TensileYieldingCalculator,
    TensileRuptureCalculator,
    TensileYieldWhitmore,
    CompressionBucklingCalculator,
    PlateTensileYieldingCalculator,
    WebLocalYieldingCalculator,
    WebLocalCrippingCalculator,
    ShearYieldingCalculator,
    PryingActionCalculator,
    WeldCalculator,
    ConnectionAnalysis,
    aisc_360_14th
)
from steel_lib.core import DesignCode

In [2]:
beam = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W21X83",
    material=MATERIALS["a992"],
    shape_type="W",length=25 * si.ft,
    role="BEAM"

)
support = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W14X90",
    material=MATERIALS["a992"],
    shape_type="W",
    role="COLUMN"

)
brace = MemberFactory.create_steelpy_member(
    section_class=aisc.L_shapes,
    section_name="L8X6X1",
    material=MATERIALS["a36"],
    shape_type="L",
    loading_condition=2,  # Assuming this is a bracing member
    role="BRACE"
)
end_plate_column = Plate(
    t=5/8 * si.inch,
    material=MATERIALS["a572_gr50"],
    width=10 * si.inch,
)


In [3]:
loads = GlobalLoads(fx=0 * si.kip, fy=0 * si.kip,direct_load=840*si.kip)

In [4]:
bolt_1 = BoltConfiguration(row_spacing=3.0 * si.inch,
        column_spacing=3.0 * si.inch,
        n_rows=2,
        n_columns=7,
        edge_distance_vertical=2 * si.inch,
        edge_distance_horizontal=1.5 * si.inch,
        bolt_diameter=7/8 * si.inch,
        bolt_grade=BOLT_GRADES["a325_x"],
        angle=47.2 * math.pi / 180,)

brace_gusset_connection = ConnectionFactory.create_bolted_connection(
        member_a=brace,
        member_b=support,
        component_a=ConnectionComponent.TOTAL,
        component_b=ConnectionComponent.WEB,
        # role_a="BEAM",
        # role_b="COLUMN",
        connection_configuration = bolt_1,
        global_loads=loads
    )

In [5]:
aisc_360_14th
checkeraasd =  DesignCode(aisc_360_14th,debug=True)


In [6]:
asdasd = checkeraasd.check_limit_states(brace_gusset_connection)    
import pandas as pd
df = pd.DataFrame(asdasd)
df

BoltConfiguration(row_spacing=3.000 inch, column_spacing=3.000 inch, edge_distance_vertical=2.000 inch, edge_distance_horizontal=1.500 inch, bolt_diameter=0.875 inch, bolt_grade=BoltGrade(name='A325-X', Fnt=90.000 ksi, Fnv=68.000 ksi), angle=0.8237954069413236, n_rows=2, n_columns=7, connection_type='bolted')


<IPython.core.display.Latex object>

<IPython.core.display.Latex object>


--- DEBUG: Bolt Shear Strength ---
  Inputs:
    Nominal Shear Stress (Fnv)         : 68.000 ksi
    Bolt Area (Ab)                     : 0.601 inch²
    Number of Shear Planes             : 2
    Resistance Factor (phi)            : 0.7500
  Output:
                                         --------------------
    Design Strength (phiRn)            : 61.335 kip
--- END DEBUG: Bolt Shear Strength ---


<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

,name,demand,capacity,dcr,remarks,details
0,Bolt Shear Strength,60.000 kip,61.335 kip,0.978239,remarks.PASS,None
1,Bolt Tensile Strength,0.000 kip,-26.646 kip,-0.000000,remarks.PASS,None
2,Tensile Yielding (TOTAL),840.000 kip,848.880 kip,0.989539,remarks.PASS,None
3,Tensile Rupture (TOTAL),840.000 kip,877.177 kip,0.957617,remarks.PASS,None


In [7]:
brace_gusset_connection.member_a

ConnectionEndpoint(member=<steelpy.steelpy.Section object at 0x0000022AE59655B0>, component=<ConnectionComponent.TOTAL: 'total'>, role='BRACE', loads=DesignLoads(Pu=840.000 kip, Vu=0.000 kip, Aub=0.000 kip, out_of_plane_force=0.000 kip), connection_configuration=BoltConfiguration(row_spacing=3.000 inch, column_spacing=3.000 inch, edge_distance_vertical=2.000 inch, edge_distance_horizontal=1.500 inch, bolt_diameter=0.875 inch, bolt_grade=BoltGrade(name='A325-X', Fnt=90.000 ksi, Fnv=68.000 ksi), angle=0.8237954069413236, n_rows=2, n_columns=7, connection_type='bolted'), design_method='LRFD', shear_condition=2)

In [8]:
brace_gusset_connection.member_a.loads.Vu

0.000 kip

In [9]:
getattr(brace_gusset_connection.member_a.loads,"Pu",0.0)

840.000 kip

In [10]:
asdasd = BoltShearCalculator(brace_gusset_connection.member_a)

BoltConfiguration(row_spacing=3.000 inch, column_spacing=3.000 inch, edge_distance_vertical=2.000 inch, edge_distance_horizontal=1.500 inch, bolt_diameter=0.875 inch, bolt_grade=BoltGrade(name='A325-X', Fnt=90.000 ksi, Fnv=68.000 ksi), angle=0.8237954069413236, n_rows=2, n_columns=7, connection_type='bolted')


<IPython.core.display.Latex object>

asdasd


In [12]:
from steel_lib.calculations import area_circular_bolt
asdasd = area_circular_bolt(7/8 * si.inch)

<IPython.core.display.Latex object>

In [13]:
asdasd

0.601 inch²

In [14]:
}

SyntaxError: unmatched '}' (70646322.py, line 1)